# CIFAR-10 CNN: HOOI Tucker-2

`02_hooi.ipynb` の HOOI を `conv2` の partial Tucker-2 に適用し、同じ rank の HOSVD と比較する。
03までの baseline / split / 自動選択 rank を引き継ぐ。今回は fine-tuning せず、**HOOI 自体の効果**を見る。

重要な練習部分は `tucker2_hooi_sweep()` の mode 0 / 1 更新だけ。その他の実験処理は完成形。

In [ ]:
from __future__ import annotations
from pathlib import Path
import copy, sys
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (p / "src/nn_compression").is_dir():
        sys.path.insert(0, str(p / "src"))
        break

from nn_compression.compression import build_tucker2_conv, hosvd, reconstruct_tucker, truncated_svd
from nn_compression.datasets import shuffled_index_splits
from nn_compression.metrics import count_parameters, parameters_reduction, relative_frobenius_error
from nn_compression.models import CIFAR10CNN
from nn_compression.tensor import mode_dot, unfold
from nn_compression.training import evaluate
from nn_compression.utils import find_project_root, get_experiment_dirs, set_seed

root = find_project_root(Path.cwd())
_, _, results_dir = get_experiment_dirs(root, "20_tucker", "10_cifar10_cnn", "04_hooi_tucker2")
selected_csv = root / "results/20_tucker/10_cifar10_cnn/02_rank_sweep/selected_rank_settings.csv"
ft_csv = root / "results/20_tucker/10_cifar10_cnn/03_finetuning/finetuning_comparison.csv"
model_path = root / "models/10_svd/40_cifar10_cnn/02_svd_global_compression_using_src_corrected/cifar10_cnn_baseline.pt"

SEED = 0
set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

## 1. 03と同じ評価条件を準備

rank は 02 の `selected_rank_settings.csv` の **balanced** を使う。ここでは rank search しない。
HOOI の反復停止は weight reconstruction error だけで決め、validation/test を収束判定に使わない。

In [ ]:
evaluation_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
full_train = datasets.CIFAR10(root=root / "data", train=True, download=False, transform=evaluation_transform)
test_dataset = datasets.CIFAR10(root=root / "data", train=False, download=False, transform=evaluation_transform)

_, _, val_rank_idx = shuffled_index_splits(len(full_train), (40_000, 5_000, 5_000), seed=SEED)
val_rank = Subset(full_train, val_rank_idx)
val_loader = DataLoader(val_rank, batch_size=256, shuffle=False, num_workers=0, pin_memory=(device.type=="cuda"))
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=0, pin_memory=(device.type=="cuda"))

if not selected_csv.is_file():
    raise FileNotFoundError(f"{selected_csv}\n先に 02_rank_sweep.ipynb を実行してください。")
selected = pd.read_csv(selected_csv)
row = selected[selected["role"] == "balanced"]
if len(row) != 1:
    raise ValueError("balanced は1行である必要があります。")
row = row.iloc[0]
rank_out, rank_in = int(row["rank_out"]), int(row["rank_in"])

baseline = CIFAR10CNN().to(device)
ckpt = torch.load(model_path, map_location=device, weights_only=False)
baseline.load_state_dict(ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt)
baseline.eval()
criterion = nn.CrossEntropyLoss()

base_val_loss, base_val_acc = evaluate(baseline, val_loader, criterion, device)
base_test_loss, base_test_acc = evaluate(baseline, test_loader, criterion, device)
print("balanced rank:", (rank_out, rank_in))
print("baseline:", base_val_acc, base_test_acc)

## 2. HOSVD Tucker-2 を基準にする

`conv2.weight = (Cout, Cin, kH, kW)` の mode 0/1 だけを `(rank_out, rank_in)` に圧縮する。
HOOIでも同じrankを使うので、parameters / MACs は変わらない。

In [ ]:
weight = baseline.conv2.weight.detach()
ranks = {0: rank_out, 1: rank_in}

core_h, factors_h = hosvd(weight, ranks)
hat_h = reconstruct_tucker(core_h, factors_h)
err_h = relative_frobenius_error(weight, hat_h)

model_h = copy.deepcopy(baseline)
model_h.conv2 = build_tucker2_conv(model_h.conv2, rank_out, rank_in).to(device)
h_val_loss, h_val_acc = evaluate(model_h, val_loader, criterion, device)
h_test_loss, h_test_acc = evaluate(model_h, test_loader, criterion, device)

print("weight:", tuple(weight.shape))
print("core:", tuple(core_h.shape))
print("HOSVD weight error:", float(err_h))
print("HOSVD val/test:", h_val_acc, h_test_acc)

## 3. 【重要】partial HOOI の1 sweepを実装

`U_out` 更新:
\[
W \times_1 U_{in}^{T} \rightarrow \mathrm{unfold}_{0} \rightarrow \mathrm{SVD} \rightarrow U_{out}
\]

`U_in` 更新:
\[
W \times_0 U_{out}^{T} \rightarrow \mathrm{unfold}_{1} \rightarrow \mathrm{SVD} \rightarrow U_{in}
\]

**mode 1 更新では、この sweep で更新済みの新しい `U_out` を使う。**
mode 2/3 は圧縮しない。

In [ ]:
def tucker2_hooi_sweep(weight, factors, rank_out, rank_in):
    updated = {mode: U.clone() for mode, U in factors.items()}

    # TODO 1: U_out (mode 0)
    # projected_out = mode_dot(weight, updated[1].T, 1)
    # unfold(projected_out, 0) を truncated_svd(..., rank_out)
    # 左特異ベクトルを updated[0] へ。

    # TODO 2: U_in (mode 1)
    # projected_in = mode_dot(weight, updated[0].T, 0)
    # ↑ updated[0] は TODO 1 で更新済みのものを使う。
    # unfold(projected_in, 1) を truncated_svd(..., rank_in)
    # 左特異ベクトルを updated[1] へ。

    raise NotImplementedError("TODO: partial HOOI の mode 0 / 1 update を実装")
    # return updated

## 4. HOOIの反復・core計算

ここは完成形。HOSVDを初期値にして sweep を繰り返し、weight relative error の改善量が `tol` 未満なら終了する。

In [ ]:
def core_from_factors(weight, factors):
    core = mode_dot(weight, factors[0].T, 0)
    return mode_dot(core, factors[1].T, 1)

def tucker2_hooi(weight, rank_out, rank_in, max_iters=20, tol=1e-6):
    _, factors = hosvd(weight, {0: rank_out, 1: rank_in})
    factors = {m: U.clone() for m, U in factors.items()}
    history, prev = [], None

    for it in range(1, max_iters + 1):
        factors = tucker2_hooi_sweep(weight, factors, rank_out, rank_in)
        core = core_from_factors(weight, factors)
        hat = reconstruct_tucker(core, factors)
        error = float(relative_frobenius_error(weight, hat))
        history.append({"iteration": it, "weight_relative_error": error})
        if prev is not None and 0.0 <= prev - error < tol:
            break
        prev = error

    return core_from_factors(weight, factors), factors, pd.DataFrame(history)

core_o, factors_o, history = tucker2_hooi(weight, rank_out, rank_in)
hat_o = reconstruct_tucker(core_o, factors_o)
err_o = relative_frobenius_error(weight, hat_o)

print("HOSVD:", float(err_h))
print("HOOI :", float(err_o))
history

## 5. HOOI factor を同じ Tucker-2 Conv に入れて評価

1x1 → kxk → 1x1 の組み立ては 01 で学習済みなので完成形。
HOSVD/HOOIで違うのは `core / U_out / U_in` の値だけ。

In [ ]:
def build_from_components(conv, core, u_out, u_in):
    factory = {"device": conv.weight.device, "dtype": conv.weight.dtype}
    r_out, r_in = u_out.shape[1], u_in.shape[1]

    left = nn.Conv2d(conv.in_channels, r_in, 1, bias=False, **factory)
    middle = nn.Conv2d(r_in, r_out, conv.kernel_size, stride=conv.stride,
                       padding=conv.padding, dilation=conv.dilation,
                       bias=False, padding_mode=conv.padding_mode, **factory)
    right = nn.Conv2d(r_out, conv.out_channels, 1, bias=(conv.bias is not None), **factory)

    with torch.no_grad():
        left.weight.copy_(u_in.T[:, :, None, None])
        middle.weight.copy_(core)
        right.weight.copy_(u_out[:, :, None, None])
        if conv.bias is not None:
            right.bias.copy_(conv.bias)
    return nn.Sequential(left, middle, right)

model_o = copy.deepcopy(baseline)
model_o.conv2 = build_from_components(model_o.conv2, core_o, factors_o[0], factors_o[1]).to(device)
o_val_loss, o_val_acc = evaluate(model_o, val_loader, criterion, device)
o_test_loss, o_test_acc = evaluate(model_o, test_loader, criterion, device)

assert count_parameters(model_h) == count_parameters(model_o)
reduction = parameters_reduction(baseline, model_o)

df = pd.DataFrame([
    ["baseline", None, base_val_loss, base_val_acc, base_test_loss, base_test_acc, count_parameters(baseline), 0.0],
    ["hosvd_tucker2", float(err_h), h_val_loss, h_val_acc, h_test_loss, h_test_acc, count_parameters(model_h), reduction],
    ["hooi_tucker2", float(err_o), o_val_loss, o_val_acc, o_test_loss, o_test_acc, count_parameters(model_o), reduction],
], columns=["method","weight_relative_error","validation_loss","validation_acc","test_loss","test_acc","parameters","parameters_reduction"])

df["rank_out"] = [None, rank_out, rank_out]
df["rank_in"] = [None, rank_in, rank_in]
df["all_macs_reduction"] = [None, row.get("all_macs_reduction", None), row.get("all_macs_reduction", None)]
df

## 6. 03の結果を参照して保存

03の `balanced` fine-tuning 結果があれば参考表示する。ここでは HOOI を fine-tuning しない。
まず圧縮直後の HOSVD vs HOOI を分離して比較する。

In [ ]:
if ft_csv.is_file():
    ft = pd.read_csv(ft_csv)
    display(ft[ft["role"] == "balanced"])

results_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(results_dir / "hooi_vs_hosvd.csv", index=False)
history.to_csv(results_dir / "hooi_error_history.csv", index=False)
print("saved:", results_dir)

## 完了条件

- [ ] 02の balanced rank を自動取得
- [ ] HOSVDの weight error / accuracy を確認
- [ ] `tucker2_hooi_sweep` を自分で実装
- [ ] HOOIの誤差推移を確認
- [ ] 同rankで HOSVD / HOOI の parameters が同じことを確認
- [ ] 圧縮直後 validation/test accuracy を比較
- [ ] 03の fine-tuning 結果は参考値として分離